### Commercial (ComStock) reconstruction — IECC Zone 2A

Mirrors `residential_2A.ipynb` (same weather blend, same feature set, same LightGBM + blocked
seasonal CV methodology) but for the 15 ComStock commercial archetypes, following the original
`commercial.ipynb` Coast pipeline for the employment -> sqft -> bus steps (CBP/ZBP employment,
`get_comstock_type()` NAICS mapping, CBECS sqft/employee factors), generalized to zone 2A using
the new statewide ZCTA -> bus mapping (`zcta_bus_mapping_tx_final.csv`, processed in
`00_zone_mapping.ipynb` into `zcta_to_zone.csv`).

Depends on `00_zone_mapping.ipynb` (county/bus/ZCTA -> zone, weather weights) having been run first.
Reuses the statewide ZBP employment file (`loadcurves/exports/tx_zbp_2018_commercial_raw.csv`,
already covers all TX ZIP prefixes) rather than re-downloading the ~60MB ZBP archive per zone.

In [1]:
# env1 3.11.13 (repo default)
import pandas as pd
import numpy as np
from pathlib import Path
import requests
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_squared_error

ZONE = "2A"
ZONE_PREFIX = "2a"

exports = Path("../../loadcurves/exports")
statewide_dir = Path("..")
weather_dir = statewide_dir / "weather_by_county"
weather_dir.mkdir(exist_ok=True)
comstock_raw_dir = statewide_dir / "comstock_raw" / ZONE
comstock_raw_dir.mkdir(parents=True, exist_ok=True)
models_dir = statewide_dir / "models" / ZONE
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir = statewide_dir / "metrics"
metrics_dir.mkdir(exist_ok=True)
exports_out_dir = statewide_dir / "exports"
exports_out_dir.mkdir(exist_ok=True)

bus_to_zone = pd.read_csv(statewide_dir / "bus_zone_master.csv", dtype={"COUNTYFP": str})
bus_to_zone["COUNTYFP"] = bus_to_zone["COUNTYFP"].str.zfill(3)

weights_df = pd.read_csv(statewide_dir / "zone_county_weather_weights.csv", dtype={"county_fips": str})
zone_weights = weights_df[weights_df["iecc_zone"] == ZONE].set_index("county_fips")["weight"]

# Area-split ZCTA -> bus weights (100 equal-area parts per ZCTA, built by 00_zone_mapping.ipynb).
# Replaces the nearest-bus zcta_to_zone join, which reached only 1,598 buses statewide.
# Rows are selected by the *bus's* zone, so a ZCTA straddling a zone boundary contributes its
# floor space to each side under that side's own ComStock models.
zcta_bus_w = pd.read_csv(statewide_dir / "zcta_bus_weights.csv", dtype={"ZCTA5CE10": str})
zone_zcta_bus = zcta_bus_w[zcta_bus_w["iecc_zone"] == ZONE].copy()
zone_zcta_bus["Bus_Number"] = zone_zcta_bus["Bus_Number"].astype(str)

zone_buses = bus_to_zone[bus_to_zone["iecc_zone"] == ZONE].copy()
zone_buses["Bus_Number"] = zone_buses["Bus_Number"].astype(str)

print(f"Zone {ZONE}: {zone_zcta_bus['ZCTA5CE10'].nunique()} ZCTAs over "
      f"{zone_zcta_bus['Bus_Number'].nunique()} buses ({len(zone_zcta_bus)} ZCTA-bus pairs), "
      f"{len(zone_weights)} counties")

Zone 2A: 938 ZCTAs over 3343 buses (7374 ZCTA-bus pairs), 98 counties


#### 1. Population-weighted county weather blend (same blend as the residential zone notebook)

In [2]:
weather_cols = [
    "Dry Bulb Temperature [\u00b0C]", "Relative Humidity [%]", "Wind Speed [m/s]",
    "Global Horizontal Radiation [W/m2]",
]

base_url = (
    "https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/"
    "weather/state=TX/"
)

county_weather = {}
for county_fips in zone_weights.index:
    gid = f"G48{int(county_fips) * 10:05d}"
    local_path = weather_dir / f"{gid}_2018.csv"
    if not local_path.exists():
        r = requests.get(f"{base_url}{gid}_2018.csv", timeout=60)
        r.raise_for_status()
        local_path.write_bytes(r.content)
    df = pd.read_csv(local_path)
    df["date_time"] = pd.to_datetime(df["date_time"])
    county_weather[county_fips] = df.sort_values("date_time").reset_index(drop=True).set_index("date_time")[weather_cols]

print(f"Loaded weather for {len(county_weather)}/{len(zone_weights)} zone {ZONE} counties (cached from the residential run).")

lengths = {k: len(v) for k, v in county_weather.items()}
assert len(set(lengths.values())) == 1, f"Row count mismatch across counties: {lengths}"

blended = sum(county_weather[c] * zone_weights[c] for c in county_weather).reset_index()
print(f"Blended weather shape: {blended.shape}")

Loaded weather for 98/98 zone 2A counties (cached from the residential run).
Blended weather shape: (8760, 5)


#### 2. Feature engineering (cyclical calendar + lagged/rolling temperature — same winning set)

In [3]:
safe_names = {
    "Dry Bulb Temperature [\u00b0C]": "Temp_C",
    "Relative Humidity [%]": "RH_pct",
    "Wind Speed [m/s]": "WindSpeed_mps",
    "Global Horizontal Radiation [W/m2]": "GHI_Wm2",
}
w = blended.rename(columns=safe_names).copy()
weather_cols_safe = [safe_names[c] for c in weather_cols]

w["hour"] = w["date_time"].dt.hour
w["month"] = w["date_time"].dt.month
w["day_of_week"] = w["date_time"].dt.dayofweek
w["month_sin"] = np.sin(2 * np.pi * w["month"] / 12)
w["month_cos"] = np.cos(2 * np.pi * w["month"] / 12)
w["hour_sin"] = np.sin(2 * np.pi * w["hour"] / 24)
w["hour_cos"] = np.cos(2 * np.pi * w["hour"] / 24)
w["dow_sin"] = np.sin(2 * np.pi * w["day_of_week"] / 7)
w["dow_cos"] = np.cos(2 * np.pi * w["day_of_week"] / 7)

w["Temp_lag1h"] = w["Temp_C"].shift(1).bfill()
w["Temp_lag3h"] = w["Temp_C"].shift(3).bfill()
w["Temp_roll24h"] = w["Temp_C"].rolling(window=24, min_periods=1).mean()

feature_cols = weather_cols_safe + [
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "Temp_lag1h", "Temp_lag3h", "Temp_roll24h",
]

weather_clean = w
X = w[feature_cols]
print(f"X shape: {X.shape}")

X shape: (8760, 13)


#### 3. Download & normalize the 15 ComStock archetype profiles for zone 2A

In [4]:
COM_TARGET_COL = "out.electricity.total.energy_consumption.kwh"

com_building_types = [
    "fullservicerestaurant", "grocery", "hospital", "largehotel", "largeoffice",
    "mediumoffice", "outpatient", "primaryschool", "quickservicerestaurant",
    "retailstandalone", "retailstripmall", "secondaryschool", "smallhotel",
    "smalloffice", "warehouse",
]

comstock_base_url = (
    "https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2025/comstock_amy2018_release_3/"
    "timeseries_aggregates/by_ashrae_iecc_climate_zone_2006/upgrade=0/"
    f"ashrae_iecc_climate_zone_2006={ZONE}/"
)


def aggregate_to_hourly_per_sqft(df_15min, target_col):
    df = df_15min.copy()
    time_col = "timestamp" if "timestamp" in df.columns else "date_time"
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.set_index(time_col)

    total_sqft = df["floor_area_represented"].iloc[0] if "floor_area_represented" in df.columns else 1.0
    if not total_sqft:
        total_sqft = 1.0

    df_hourly = df[[target_col]].resample("h", label="right", closed="right").sum()
    if not df_hourly.empty and df_hourly.index[0].hour == 0 and df_hourly.index[0].minute == 0:
        df_hourly = df_hourly.iloc[1:]

    df_hourly[target_col] = df_hourly[target_col] / total_sqft
    return df_hourly


hourly_profiles = {}
for btype in com_building_types:
    fname = f"up0-{ZONE_PREFIX}-{btype}.csv"
    local_path = comstock_raw_dir / fname
    if not local_path.exists():
        r = requests.get(comstock_base_url + fname, timeout=120)
        r.raise_for_status()
        local_path.write_bytes(r.content)
    raw = pd.read_csv(local_path)
    hourly_profiles[btype] = aggregate_to_hourly_per_sqft(raw, COM_TARGET_COL)

for name, d in hourly_profiles.items():
    if len(d) != len(X):
        print(f"  !! length mismatch for {name}: {len(d)} rows vs X {len(X)} rows")
print(f"Loaded {len(hourly_profiles)} commercial archetype profiles for zone {ZONE}.")

Loaded 15 commercial archetype profiles for zone 2A.


#### 4. Honest evaluation: blocked seasonal 4-fold CV (LightGBM)

In [5]:
SEASON_MONTHS = {"winter": [12, 1, 2], "spring": [3, 4, 5], "summer": [6, 7, 8], "fall": [9, 10, 11]}


def seasonal_blocked_cv(X, y, months_series):
    months_arr = months_series.values
    scores = {}
    for season, months in SEASON_MONTHS.items():
        test_mask = np.isin(months_arr, months)
        X_train, X_test = X[~test_mask], X[test_mask]
        y_train, y_test = y[~test_mask], y[test_mask]
        model = LGBMRegressor(n_estimators=100, max_depth=-1, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        scores[season] = r2_score(y_test, preds)
    return scores


cv_results = []
for b_type, df_hourly in hourly_profiles.items():
    y = df_hourly[COM_TARGET_COL].values
    season_scores = seasonal_blocked_cv(X, y, weather_clean["month"])
    mean_r2 = np.mean(list(season_scores.values()))
    cv_results.append({"Zone": ZONE, "Building_Type": b_type, "Mean_R2": mean_r2, **season_scores})
    print(f" -> {b_type:25s} mean R2={mean_r2:.4f}  " + "  ".join(f"{s}={v:.3f}" for s, v in season_scores.items()))

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(metrics_dir / f"seasonal_cv_comstock_{ZONE}.csv", index=False)

 -> fullservicerestaurant     mean R2=0.9568  winter=0.938  spring=0.945  summer=0.982  fall=0.962


 -> grocery                   mean R2=0.9359  winter=0.881  spring=0.938  summer=0.979  fall=0.946


 -> hospital                  mean R2=0.9654  winter=0.967  spring=0.931  summer=0.989  fall=0.975


 -> largehotel                mean R2=0.8785  winter=0.784  spring=0.899  summer=0.922  fall=0.909


 -> largeoffice               mean R2=0.9631  winter=0.946  spring=0.969  summer=0.980  fall=0.957


...

 -> retailstripmall           mean R2=0.9553  winter=0.934  spring=0.945  summer=0.984  fall=0.958


 -> secondaryschool           mean R2=0.9204  winter=0.923  spring=0.950  summer=0.924  fall=0.884


 -> smallhotel                mean R2=0.6909  winter=0.800  spring=0.745  summer=0.723  fall=0.495


 -> smalloffice               mean R2=0.9295  winter=0.891  spring=0.934  summer=0.972  fall=0.921


 -> warehouse                 mean R2=0.9372  winter=0.914  spring=0.941  summer=0.960  fall=0.934


#### 5. Train final production models on the full year

In [6]:
for b_type, df_hourly in hourly_profiles.items():
    y = df_hourly[COM_TARGET_COL].values
    model = LGBMRegressor(n_estimators=100, max_depth=-1, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X, y)
    safe_name = b_type.replace(" ", "_").replace("/", "-")
    joblib.dump(model, models_dir / f"comstock_model_{safe_name}.joblib")

print(f"{len(hourly_profiles)} final commercial models saved to {models_dir}")

15 final commercial models saved to ../models/2A


#### 6. Employment -> sqft -> bus (zone 2A), from CBP/ZBP

Reuses the shared statewide ZBP raw file (all TX ZIP prefixes already filtered), restricts to
this zone's ZCTAs via `zcta_to_zone.csv`, then applies the same NAICS -> ComStock-type mapping
and CBECS-derived sqft/employee factors as the original Coast `commercial.ipynb`.

In [7]:
zbp = pd.read_csv(exports / "tx_zbp_2018_commercial_raw.csv", dtype={"zip": str, "naics": str})
zbp_clean = zbp[~zbp["naics"].str.contains("[-/]")].copy()
zbp_clean["naics"] = zbp_clean["naics"].astype(int)
zbp_clean["Final_Employees"] = pd.to_numeric(zbp_clean["Final_Employees"]).fillna(0).astype(int)

zone_zips = zone_zcta_bus["ZCTA5CE10"].unique().tolist()
zbp_zone = zbp_clean[zbp_clean["zip"].isin(zone_zips)].copy()
print(f"Zone {ZONE}: {len(zbp_zone)} ZBP rows across {zbp_zone['zip'].nunique()} ZCTAs with ZBP data (of {len(zone_zips)} zone ZCTAs)")


def get_comstock_type(code):
    if code == 624410: return "primaryschool"
    if code == 713940: return "retailstandalone"
    if code == 445110: return "grocery"
    if code == 722511: return "fullservicerestaurant"
    if code == 722513: return "quickservicerestaurant"
    if code == 722515: return "quickservicerestaurant"

    if code // 10000 == 21: return "Industrial"
    if code // 10000 in [31, 32, 33]: return "Industrial"
    if code // 10000 == 11: return "Industrial"
    if code // 10000 == 22: return "Industrial"

    prefix_4 = code // 100
    prefix_3 = code // 1000
    prefix_2 = code // 10000

    if prefix_4 == 7211: return "largehotel"
    if prefix_4 == 7212: return "smallhotel"
    if prefix_4 == 6111: return "primaryschool"
    if prefix_4 == 6112: return "secondaryschool"
    if prefix_4 == 6113: return "secondaryschool"
    if prefix_3 == 622: return "hospital"
    if prefix_3 == 621: return "outpatient"
    if prefix_3 == 623: return "outpatient"
    if prefix_3 == 624: return "outpatient"
    if prefix_3 == 722: return "fullservicerestaurant"
    if prefix_2 in [51, 52, 53, 54, 55, 56, 92]: return "mediumoffice"
    if prefix_2 in [44, 45, 71]: return "retailstandalone"
    if prefix_2 in [42, 48, 49]: return "warehouse"
    if prefix_2 == 81: return "smalloffice"
    if prefix_2 == 23: return "smalloffice"
    return "Other"


zbp_zone["ComStock_Type"] = zbp_zone["naics"].apply(get_comstock_type)
df_commercial = zbp_zone[(zbp_zone["ComStock_Type"] != "Industrial") & (zbp_zone["ComStock_Type"] != "Other")].copy()
df_industrial = zbp_zone[zbp_zone["ComStock_Type"] == "Industrial"].copy()
df_commercial = df_commercial[df_commercial["Final_Employees"] > 0].copy()

print(f"Zone {ZONE} commercial employees: {df_commercial['Final_Employees'].sum():,.0f} (mapped to {df_commercial['ComStock_Type'].nunique()} archetypes)")
print(f"Zone {ZONE} industrial employees set aside: {df_industrial['Final_Employees'].sum():,.0f}")

Zone 2A: 30136 ZBP rows across 690 ZCTAs with ZBP data (of 938 zone ZCTAs)
Zone 2A commercial employees: 1,595,500 (mapped to 13 archetypes)
Zone 2A industrial employees set aside: 31,412


In [8]:
sqft_per_emp = {
    "quickservicerestaurant": 300, "fullservicerestaurant": 400,
    "grocery": 750, "retailstandalone": 800, "retailstripmall": 700,
    "hospital": 700, "outpatient": 500,
    "largehotel": 1500, "smallhotel": 1200,
    "primaryschool": 1100, "secondaryschool": 1300,
    "largeoffice": 200, "mediumoffice": 200, "smalloffice": 200,
    "warehouse": 3000,
}

all_comstock_types = [
    "smalloffice", "mediumoffice", "largeoffice",
    "retailstandalone", "retailstripmall", "grocery",
    "primaryschool", "secondaryschool",
    "outpatient", "hospital",
    "smallhotel", "largehotel",
    "quickservicerestaurant", "fullservicerestaurant",
    "warehouse",
]

df_commercial["SqFt_Factor"] = df_commercial["ComStock_Type"].map(sqft_per_emp).fillna(250)
df_commercial["Total_SqFt"] = df_commercial["Final_Employees"] * df_commercial["SqFt_Factor"]

pivot_sqft = df_commercial.pivot_table(
    index="zip", columns="ComStock_Type", values="Total_SqFt", aggfunc="sum", fill_value=0,
)
pivot_sqft = pivot_sqft.reindex(columns=all_comstock_types, fill_value=0)
print(f"Zone {ZONE} total commercial sqft: {pivot_sqft.values.sum() / 1e6:,.2f} million sqft")

pivot_sqft.index = pivot_sqft.index.astype(str)
sqft_flat = pivot_sqft.reset_index().rename(columns={"zip": "ZCTA5CE10"})

# Area-split allocation: a ZCTA's floor space is spread over every bus serving part of it,
# weighted by the share of its 100 equal-area parts that bus won.
merged_sqft = pd.merge(zone_zcta_bus[["ZCTA5CE10", "Bus_Number", "weight"]], sqft_flat,
                       on="ZCTA5CE10", how="left")
merged_sqft[all_comstock_types] = merged_sqft[all_comstock_types].fillna(0.0).mul(merged_sqft["weight"], axis=0)

bus_sqft_matrix = (merged_sqft.groupby("Bus_Number")[all_comstock_types].sum()
                   .reindex(zone_buses["Bus_Number"].values).fillna(0.0))
bus_sqft_matrix.index.name = "Bus_Number"
n_zero = (bus_sqft_matrix.sum(axis=1) == 0).sum()
print(f"Zone {ZONE}: {len(bus_sqft_matrix)} buses, {n_zero} with zero commercial sqft; "
      f"{bus_sqft_matrix.values.sum()/1e6:,.2f} M sqft allocated "
      f"({bus_sqft_matrix.values.sum()/pivot_sqft.values.sum():.2%} of the zone's ZBP sqft retained)")

bus_sqft_matrix.to_csv(exports_out_dir / f"bus_commercial_sqft_{ZONE}.csv")

Zone 2A total commercial sqft: 810.41 million sqft
Zone 2A: 3580 buses, 346 with zero commercial sqft; 804.03 M sqft allocated (99.21% of the zone's ZBP sqft retained)


#### 7. Predict & assemble bus-level commercial profiles (zone 2A)

In [9]:
base_profiles = {}
for b_type in all_comstock_types:
    safe_name = b_type.replace(" ", "_").replace("/", "-")
    model = joblib.load(models_dir / f"comstock_model_{safe_name}.joblib")
    base_profiles[b_type] = model.predict(X)

base_profiles_df = pd.DataFrame(base_profiles, index=weather_clean["date_time"])

counts_matrix = bus_sqft_matrix.reindex(columns=base_profiles_df.columns).fillna(0)
bus_profiles_df = base_profiles_df.dot(counts_matrix.T)

out_path = exports_out_dir / f"commercial_bus_load_profiles_8760_{ZONE}.csv"
bus_profiles_df.to_csv(out_path)
print(f"Zone {ZONE} commercial bus profiles: {bus_profiles_df.shape} -> {out_path}")
print(f"Peak (sum of all buses): {(bus_profiles_df.sum(axis=1)/1000).max():,.1f} MW (uncalibrated)")

Zone 2A commercial bus profiles: (8760, 3580) -> ../exports/commercial_bus_load_profiles_8760_2A.csv
Peak (sum of all buses): 5,305.7 MW (uncalibrated)


#### Summary — Zone 2A

Seasonal CV table saved to `metrics/seasonal_cv_comstock_2A.csv`, {15} production models saved to
`models/2A/comstock_model_*.joblib`, bus-level commercial sqft matrix saved to
`exports/bus_commercial_sqft_2A.csv`, bus-level commercial load profiles saved to
`exports/commercial_bus_load_profiles_8760_2A.csv`. Consumed by `statewide_summary.ipynb`
for the cross-zone comparison and the Coast-county cross-check.